# SEGMENT 5 — Hyperparameter Tuning

**Project**: HDB Resale Price Prediction  
**Objective**: Find optimal hyperparameters for all models using cross-validated search  
**Primary Metric for Tuning**: RMSE (neg_root_mean_squared_error via GridSearchCV/RandomizedSearchCV)  

---

## Strategy

| Model | Search Method | Reason |
|-------|--------------|--------|
| KNN | GridSearchCV | Small parameter space (k, weights, p) |
| Ridge | GridSearchCV | Single alpha parameter |
| Lasso | GridSearchCV | Single alpha parameter |
| Elastic Net | GridSearchCV | alpha + l1_ratio (manageable grid) |
| Decision Tree | GridSearchCV | Moderate parameter space |
| Gradient Boosting | RandomizedSearchCV | Large parameter space — exhaustive too slow |

---

## Cross-Validation Design
- `KFold(n_splits=5, shuffle=True, random_state=42)`
- Scoring: `neg_root_mean_squared_error` (primary for selection)
- Refit on full training set with best params after CV
- Final evaluation on held-out test set

---
> **Rule**: The test set is NEVER used during hyperparameter search. CV runs on X_train only.

---
## Step 5.1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV, cross_val_score

print('Imports complete.')

ImportError: cannot import name 'root_mean_squared_error' from 'sklearn.metrics' (/home/auyan/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/__init__.py)

---
## Step 5.2 — Load Data

In [ ]:
X_train_A = np.load('artifacts/X_train_A.npy')
X_test_A  = np.load('artifacts/X_test_A.npy')
X_train_B = np.load('artifacts/X_train_B.npy')
X_test_B  = np.load('artifacts/X_test_B.npy')
y_train   = np.load('artifacts/y_train.npy')
y_test    = np.load('artifacts/y_test.npy')

print(f'X_train_A: {X_train_A.shape},  X_train_B: {X_train_B.shape}')
print(f'y_train  : {y_train.shape},   mean=${y_train.mean():,.0f}')

---
## Step 5.3 — Helpers

In [ ]:
CV = KFold(n_splits=5, shuffle=True, random_state=42)

def final_eval(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    mae    = mean_absolute_error(y_test, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)
    print(f'[{name}]  Test MAE=${mae:,.0f}  RMSE=${rmse:,.0f}  R²={r2:.4f}')
    return {'Model': name, 'Test MAE': round(mae,0), 'Test RMSE': round(rmse,0), 'Test R²': round(r2,4)}, y_pred

tuning_results = []

---
## Step 5.4 — Tune KNN Regressor

**Parameter grid**:
- `n_neighbors`: small k → low bias, high variance; large k → high bias, low variance
- `weights`: `uniform` (all neighbors equal) vs `distance` (closer = more weight)
- `p`: 1 = Manhattan distance, 2 = Euclidean distance

**Decision metric**: CV RMSE → choose n_neighbors that minimizes it

In [ ]:
knn_param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights'    : ['uniform', 'distance'],
    'p'          : [1, 2],
}

knn_grid = GridSearchCV(
    KNeighborsRegressor(),
    knn_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=CV, n_jobs=-1, verbose=1, refit=True
)
knn_grid.fit(X_train_A, y_train)

print(f'\nBest KNN params : {knn_grid.best_params_}')
print(f'Best CV RMSE    : ${-knn_grid.best_score_:,.0f}')

knn_tuned_result, knn_tuned_pred = final_eval('KNN (tuned)', knn_grid.best_estimator_, X_test_A, y_test)
knn_tuned_result['Best Params'] = str(knn_grid.best_params_)
knn_tuned_result['CV RMSE'] = round(-knn_grid.best_score_, 0)
tuning_results.append(knn_tuned_result)

In [ ]:
# Visualise CV results for KNN
knn_cv_df = pd.DataFrame(knn_grid.cv_results_)
knn_cv_df['mean_test_rmse'] = -knn_cv_df['mean_test_score']

fig, ax = plt.subplots(figsize=(11, 4))
for weight in ['uniform', 'distance']:
    for p_val in [1, 2]:
        mask = (knn_cv_df['param_weights'] == weight) & (knn_cv_df['param_p'] == p_val)
        subset = knn_cv_df[mask].sort_values('param_n_neighbors')
        ax.plot(subset['param_n_neighbors'], subset['mean_test_rmse'],
                marker='o', label=f'weights={weight}, p={p_val}')

ax.set_xlabel('n_neighbors')
ax.set_ylabel('CV RMSE')
ax.set_title('KNN — GridSearchCV Results', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 5.5 — Tune Ridge Regression

**Decision**: Higher alpha → stronger shrinkage → less overfitting but potentially more bias.  
CV RMSE tells us which alpha strikes the right balance.

In [ ]:
ridge_param_grid = {'alpha': [0.01, 0.1, 1, 10, 100, 500, 1000]}

ridge_grid = GridSearchCV(
    Ridge(),
    ridge_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=CV, n_jobs=-1, refit=True
)
ridge_grid.fit(X_train_A, y_train)

print(f'Best Ridge params : {ridge_grid.best_params_}')
print(f'Best CV RMSE      : ${-ridge_grid.best_score_:,.0f}')

ridge_tuned_result, ridge_tuned_pred = final_eval('Ridge (tuned)', ridge_grid.best_estimator_, X_test_A, y_test)
ridge_tuned_result['Best Params'] = str(ridge_grid.best_params_)
ridge_tuned_result['CV RMSE'] = round(-ridge_grid.best_score_, 0)
tuning_results.append(ridge_tuned_result)

---
## Step 5.6 — Tune Lasso Regression

**Secondary observation**: At optimal alpha, how many features are selected (non-zero coefficients)?  
This is a unique diagnostic for Lasso — tells you how many features the model actually "needs".

In [ ]:
lasso_param_grid = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10]}

lasso_grid = GridSearchCV(
    Lasso(max_iter=10000),
    lasso_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=CV, n_jobs=-1, refit=True
)
lasso_grid.fit(X_train_A, y_train)

print(f'Best Lasso params   : {lasso_grid.best_params_}')
print(f'Best CV RMSE        : ${-lasso_grid.best_score_:,.0f}')

best_lasso = lasso_grid.best_estimator_
nonzero_count = np.sum(np.abs(best_lasso.coef_) > 1e-6)
print(f'Features selected   : {nonzero_count} out of {len(best_lasso.coef_)}')

lasso_tuned_result, lasso_tuned_pred = final_eval('Lasso (tuned)', best_lasso, X_test_A, y_test)
lasso_tuned_result['Best Params'] = str(lasso_grid.best_params_)
lasso_tuned_result['CV RMSE'] = round(-lasso_grid.best_score_, 0)
tuning_results.append(lasso_tuned_result)

---
## Step 5.7 — Tune Elastic Net

In [ ]:
en_param_grid = {
    'alpha'   : [0.0001, 0.001, 0.01, 0.1, 1],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
}

en_grid = GridSearchCV(
    ElasticNet(max_iter=10000),
    en_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=CV, n_jobs=-1, refit=True
)
en_grid.fit(X_train_A, y_train)

print(f'Best ElasticNet params : {en_grid.best_params_}')
print(f'Best CV RMSE           : ${-en_grid.best_score_:,.0f}')

en_tuned_result, en_tuned_pred = final_eval('Elastic Net (tuned)', en_grid.best_estimator_, X_test_A, y_test)
en_tuned_result['Best Params'] = str(en_grid.best_params_)
en_tuned_result['CV RMSE'] = round(-en_grid.best_score_, 0)
tuning_results.append(en_tuned_result)

---
## Step 5.8 — Tune Decision Tree Regressor

**Key tuning decisions**:
- `max_depth`: primary depth limiter — lower = more regularized
- `min_samples_split` / `min_samples_leaf`: prevent tiny leaf splits → reduces overfitting
- `max_features`: controls random feature subsampling at each split (adds randomness → variance reduction)
- **Criterion always = `squared_error`** — NOT Gini/Entropy (those are for classifiers)

In [ ]:
dt_param_grid = {
    'max_depth'       : [3, 5, 8, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf' : [1, 2, 5, 10],
    'max_features'     : [None, 'sqrt', 'log2'],
}

dt_grid = GridSearchCV(
    DecisionTreeRegressor(criterion='squared_error', random_state=42),
    dt_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=CV, n_jobs=-1, verbose=1, refit=True
)
dt_grid.fit(X_train_B, y_train)

print(f'\nBest DT params : {dt_grid.best_params_}')
print(f'Best CV RMSE   : ${-dt_grid.best_score_:,.0f}')

dt_tuned_result, dt_tuned_pred = final_eval('Decision Tree (tuned)', dt_grid.best_estimator_, X_test_B, y_test)
dt_tuned_result['Best Params'] = str(dt_grid.best_params_)
dt_tuned_result['CV RMSE'] = round(-dt_grid.best_score_, 0)
tuning_results.append(dt_tuned_result)

---
## Step 5.9 — Tune Gradient Boosting Regressor (RandomizedSearchCV)

**Why RandomizedSearch here?**  
The full grid for GBR has 3×3×3×3×3×2 = 486 combinations × 5 folds = 2,430 fits.  
RandomizedSearch samples `n_iter` random combinations → much faster, comparable quality.

**Key parameters**:
- `n_estimators` + `learning_rate`: trade-off (more trees + lower rate = better generalization)
- `max_depth`: how deep each weak tree is; keep shallow (2-4) for GBR
- `subsample` < 1.0: stochastic boosting → adds randomness, reduces overfitting
- `min_samples_split` / `min_samples_leaf`: node splitting constraints

In [ ]:
gbr_param_dist = {
    'n_estimators'    : [100, 200, 300],
    'learning_rate'   : [0.01, 0.05, 0.1],
    'max_depth'       : [2, 3, 4],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 5],
    'subsample'        : [0.8, 1.0],
}

gbr_rand = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    gbr_param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=CV, n_jobs=-1, verbose=1, random_state=42, refit=True
)
gbr_rand.fit(X_train_B, y_train)

print(f'\nBest GBR params : {gbr_rand.best_params_}')
print(f'Best CV RMSE    : ${-gbr_rand.best_score_:,.0f}')

gbr_tuned_result, gbr_tuned_pred = final_eval('GBR (tuned)', gbr_rand.best_estimator_, X_test_B, y_test)
gbr_tuned_result['Best Params'] = str(gbr_rand.best_params_)
gbr_tuned_result['CV RMSE'] = round(-gbr_rand.best_score_, 0)
tuning_results.append(gbr_tuned_result)

---
## Step 5.10 — Tuned Models Comparison Table

In [ ]:
tuned_df = pd.DataFrame(tuning_results)
display_df = tuned_df[['Model','Test MAE','Test RMSE','Test R²','CV RMSE','Best Params']].copy()
display_df['Test MAE']  = display_df['Test MAE'].apply(lambda x: f'${float(x):,.0f}')
display_df['Test RMSE'] = display_df['Test RMSE'].apply(lambda x: f'${float(x):,.0f}')
display_df['CV RMSE']   = display_df['CV RMSE'].apply(lambda x: f'${float(x):,.0f}')

print('=== TUNED MODELS COMPARISON ===')
print(display_df.to_string(index=False))

In [ ]:
# Bar chart — tuned RMSE comparison
rmse_tuned = tuned_df['Test RMSE'].values.astype(float)
names_tuned = tuned_df['Model'].tolist()

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(names_tuned)))
bars = ax.bar(names_tuned, rmse_tuned, color=colors, edgecolor='white')

for bar, val in zip(bars, rmse_tuned):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'${val/1e3:.1f}K', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Test RMSE (SGD)')
ax.set_title('Tuned Models — Test RMSE Comparison', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 5.11 — Save Tuned Models and Results

In [ ]:
joblib.dump(knn_grid.best_estimator_,  'artifacts/model_knn_tuned.pkl')
joblib.dump(ridge_grid.best_estimator_,'artifacts/model_ridge_tuned.pkl')
joblib.dump(lasso_grid.best_estimator_,'artifacts/model_lasso_tuned.pkl')
joblib.dump(en_grid.best_estimator_,   'artifacts/model_en_tuned.pkl')
joblib.dump(dt_grid.best_estimator_,   'artifacts/model_dt_tuned.pkl')
joblib.dump(gbr_rand.best_estimator_,  'artifacts/model_gbr_tuned.pkl')

tuned_df.to_csv('artifacts/results_tuned.csv', index=False)

print('Saved all tuned models and results_tuned.csv')

---
## ✅ Segment 5 Complete

**Summary**:
- All 6 models have been tuned using 5-fold CV with RMSE as the selection metric
- Best parameters are saved and models are refitted on full training data
- GBR expected to be the leading individual model

**Next**: Proceed to `06_ensemble_final.ipynb`
- Build VotingRegressor combining best models
- Compare ensemble vs individual best
- Final model selection and business interpretation
- Feature importance summary